## 0) Setup & Data Load

In [ ]:
RESTAURANT_KEYWORDS = ["Chipotle"]

## 1) Restaurant Selection

In [ ]:
RESTAURANT_KEYWORDS = ["Chipotle"]

## 2) Data Extraction: all inspection & violation rows for the chosen restaurant

In [ ]:
!curl 'https://data.cityofnewyork.us/api/views/43nn-pn8j/rows.csv?accessType=DOWNLOAD' -o /tmp/nyc_restaurant_inspections.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  123M    0  123M    0     0  5874k      0 --:--:--  0:00:21 --:--:-- 6320k


In [ ]:
import pandas as pd
import numpy as np


try:
    df = pd.read_csv("/tmp/nyc_restaurant_inspections.csv")
except FileNotFoundError:
    print("Make sure the dataset 'nyc_restaurant_inspections.csv' is in the /tmp directory.")
    df = None

if df is not None:

    restaurant_mask = df['DBA'].str.contains('|'.join(RESTAURANT_KEYWORDS), case=False, na=False)
    restaurant_df = df.loc[restaurant_mask].copy()

    if not restaurant_df.empty:
        chosen_zip = restaurant_df['ZIPCODE'].iloc[0]
        chosen_cuisine = restaurant_df['CUISINE DESCRIPTION'].iloc[0]
    else:
        chosen_zip = None
        chosen_cuisine = None
        print(f"No restaurants found matching keywords: {RESTAURANT_KEYWORDS}")

    print(f"Chosen Restaurant Keywords: {RESTAURANT_KEYWORDS}")
    if chosen_zip is not None:
        print(f"Inferred ZIP Code: {chosen_zip}")
    if chosen_cuisine is not None:
        print(f"Inferred Cuisine: {chosen_cuisine}")

    display(restaurant_df.head())

Chosen Restaurant Keywords: ['Chipotle']
Inferred ZIP Code: 10001.0
Inferred Cuisine: Tex-Mex


,CAMIS,DBA,BORO,BUILDING,STREET,ZIPCODE,PHONE,CUISINE DESCRIPTION,INSPECTION DATE,ACTION,...,INSPECTION TYPE,Latitude,Longitude,Community Board,Council District,Census Tract,BIN,BBL,NTA,Location Point1
554,41004963,CHIPOTLE MEXICAN GRILL,Manhattan,304,WEST 34 STREET,10001.0,2122684197,Tex-Mex,05/20/2024,Violations were cited in the following area(s).,...,Cycle Inspection / Initial Inspection,40.752390,-73.993976,104.0,3.0,10300.0,1013552.0,1.007570e+09,MN13,NaN
1058,50173397,CHIPOTLE MEXICAN GRILL #3056,Brooklyn,431,KNICKERBOCKER AVENUE,11237.0,6143187413,NaN,01/01/1900,NaN,...,NaN,40.699456,-73.919483,304.0,37.0,43100.0,3074794.0,3.032790e+09,BK77,NaN
1501,50174354,CHIPOTLE MEXICAN GRILL #3056,Brooklyn,219,HAVEMEYER STREET,11211.0,6143187413,NaN,01/01/1900,NaN,...,NaN,40.709292,-73.959000,301.0,34.0,52500.0,3063570.0,3.024600e+09,BK73,NaN
2267,50084126,CHIPOTLE MEXICAN GRILL #3044,Queens,16010,CROSSBAY BLVD,11414.0,7187381309,Mexican,04/15/2024,Violations were cited in the following area(s).,...,Cycle Inspection / Initial Inspection,40.657122,-73.839382,410.0,32.0,89200.0,4293775.0,4.140300e+09,QN57,NaN
2475,41640950,CHIPOTLE MEXICAN GRILL,Manhattan,71,SPRING STREET,10012.0,6469375202,Tex-Mex,08/18/2022,Violations were cited in the following area(s).,...,Cycle Inspection / Initial Inspection,40.722499,-73.997586,102.0,1.0,4300.0,1079952.0,1.004960e+09,MN24,NaN


## 3) Research Questions

### Q1) How many **inspections** and **violations** has this restaurant received across the dataset?

In [ ]:
q1_df = restaurant_df.copy()

inspections = q1_df["INSPECTION DATE"].dropna().nunique() if "INSPECTION DATE" in q1_df.columns else np.nan
violations = q1_df["VIOLATION CODE"].notna().sum() if "VIOLATION CODE" in q1_df.columns else np.nan

print({"total_inspections": int(inspections) if pd.notna(inspections) else None,
       "total_violations": int(violations) if pd.notna(violations) else None})

{'total_inspections': 295, 'total_violations': 942}


 The counts approximate how often the health department inspected this location and how many violations were recorded in total. A higher inspection count with relatively fewer violations suggests consistent compliance; the opposite may warrant attention.


### Q2) What are the **most frequent violation descriptions** for this restaurant?

In [ ]:
q2_df = restaurant_df.copy()
if "VIOLATION DESCRIPTION" in q2_df.columns:
    top_violations = (
        q2_df["VIOLATION DESCRIPTION"]
        .value_counts(dropna=True)
        .head(10)
        .rename_axis("VIOLATION DESCRIPTION")
        .reset_index(name="COUNT")
    )
    top_violations
else:
    print("Column VIOLATION DESCRIPTION not found.")

 These are the patterns most commonly cited by inspectors. Repeated temperature, hygiene, or pest-related issues point to process controls the restaurant should prioritize.


### Q3) How does this restaurant’s **average inspection score** compare with **same-cuisine restaurants in the same ZIP code**?

In [ ]:

if all(col in df.columns for col in ["ZIPCODE","CUISINE DESCRIPTION","SCORE"]):
    comp_mask = (df["ZIPCODE"] == chosen_zip) & (df["CUISINE DESCRIPTION"] == chosen_cuisine)
    peer_df = df.loc[comp_mask].copy()
    peer_df["SCORE"] = pd.to_numeric(peer_df["SCORE"], errors="coerce")

    this_avg = pd.to_numeric(restaurant_df["SCORE"], errors="coerce").mean()
    peer_avg = peer_df["SCORE"].mean()

    print({"restaurant_avg_score": float(this_avg) if pd.notna(this_avg) else None,
           "peer_avg_score_zip_samecuisine": float(peer_avg) if pd.notna(peer_avg) else None,
           "peer_sample_size": int(peer_df["CAMIS"].nunique()) if "CAMIS" in peer_df.columns else int(len(peer_df))})
else:
    print("Missing one of ZIPCODE / CUISINE DESCRIPTION / SCORE columns.")

{'restaurant_avg_score': 13.904328018223234, 'peer_avg_score_zip_samecuisine': 20.886363636363637, 'peer_sample_size': 6}


 Lower average scores generally indicate fewer or less severe violations . If this restaurant’s average is lower than peers in the same ZIP and cuisine, it is outperforming local competitors on inspections.


Trend: average score by year

In [ ]:
q4_df = restaurant_df.copy()
if "INSPECTION DATE" in q4_df.columns and "SCORE" in q4_df.columns:
    q4_df["INSPECTION DATE"] = pd.to_datetime(q4_df["INSPECTION DATE"], errors='coerce')
    q4_df["year"] = q4_df["INSPECTION DATE"].dt.year
    trend = (q4_df
             .pivot_table(index="year", values="SCORE", aggfunc="mean")
             .reset_index()
             .sort_values("year"))
    trend
else:
    print("Need INSPECTION DATE and SCORE to compute yearly trend.")


A downward trend over years suggests improving compliance; upward may indicate deteriorating food-safety practices or inconsistent controls.


## 4) Export filtered rows (optional for your appendix)

In [ ]:
subset_out = "filtered_rows_for_submission.csv"
restaurant_df.to_csv(subset_out, index=False)
print(f"Saved: {subset_out}")

Saved: filtered_rows_for_submission.csv
